In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
!pip install -q "tensorflow==2.20.0" pytorch-tabnet imbalanced-learn
!pip install -U ml_dtypes

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.5/44.5 kB 2.2 MB/s eta 0:00:00


In [ ]:
import sys
PROJECT_ROOT = '/content/drive/MyDrive/BurnoutGuard'
sys.path.insert(0, f'{PROJECT_ROOT}/04_src')
from config import *

In [ ]:
import random, os
import numpy as np
import tensorflow as tf

os.environ['PYTHONHASHSEED'] = str(SEED)
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

In [ ]:
import pandas as pd

df = pd.read_csv(DATA_RAW)
print(f"Shape: {df.shape}")
print(f"Missing: {df.isnull().sum().sum()}")
df.head()

Shape: (3000, 18)
Missing: 0


,Age,Gender,Study_Hours,Class_Attendance,Tuition,Exam_Frequency,Assignment_Load,Sleep_Hours,Physical_Exercise,Social_Media_Use,Screen_Time,Family_Income_Level,Peer_Pressure,Family_Support,Anxiety_Level,University_Type,Stress_Score,Stress_Level
0,24,Male,9,71,No,7,4,6,Yes,6,9,Low,6,3,5,National University,20,Medium
1,24,Female,8,43,Yes,6,8,5,Yes,6,8,Medium,3,7,8,National University,22,High
2,21,Female,4,56,Yes,3,6,5,Yes,7,9,Low,2,5,3,Private University,16,Medium
3,23,Female,4,46,No,8,6,8,No,5,6,Medium,8,6,5,Private University,16,Medium
4,20,Male,8,46,No,5,1,6,Yes,0,7,Medium,5,6,2,National University,1,Low


In [ ]:
df = df.drop(columns=DROP_COLUMNS)

print("Kolom tersisa:", df.columns.tolist())
print(f"Shape setelah drop: {df.shape}")

Kolom tersisa: ['Age', 'Gender', 'Study_Hours', 'Class_Attendance', 'Tuition', 'Exam_Frequency', 'Assignment_Load', 'Sleep_Hours', 'Physical_Exercise', 'Social_Media_Use', 'Screen_Time', 'Family_Income_Level', 'Peer_Pressure', 'Family_Support', 'Anxiety_Level', 'University_Type', 'Stress_Level']
Shape setelah drop: (3000, 17)


In [ ]:
dist = df[TARGET_COLUMN].value_counts(normalize=True)
print(dist)

Stress_Level
Medium    0.477000
Low       0.414333
High      0.108667
Name: proportion, dtype: float64


In [ ]:
X = df.drop(columns=[TARGET_COLUMN])
y = df[TARGET_COLUMN].map(CLASS_MAPPING).values
print(f"X shape: {X.shape}")
print(f"y unique: {np.unique(y)}")

X shape: (3000, 16)
y unique: [0 1 2]


In [ ]:
from sklearn.model_selection import train_test_split

X_trainval, X_test, y_trainval, y_test = train_test_split(
    X, y, test_size=TEST_SIZE, stratify=y, random_state=SEED)

X_train, X_val, y_train, y_val = train_test_split(
    X_trainval, y_trainval,
    test_size=VAL_SIZE / (1 - TEST_SIZE),
    stratify=y_trainval, random_state=SEED)

print(f"Train: {X_train.shape[0]} | Val: {X_val.shape[0]} | Test: {X_test.shape[0]}")

Train: 2099 | Val: 451 | Test: 450


In [ ]:
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer

preprocessor = ColumnTransformer([
    ('num', StandardScaler(), NUMERIC_COLS),
    ('cat', OneHotEncoder(handle_unknown='ignore',
                         sparse_output=False), CATEGORICAL_COLS),
])

X_train_p = preprocessor.fit_transform(X_train)
X_val_p   = preprocessor.transform(X_val)
X_test_p  = preprocessor.transform(X_test)

print(f"Fitur setelah OHE: {X_train_p.shape[1]}")
print(f"NaN check: {np.isnan(X_train_p).sum()}")

Fitur setelah OHE: 23
NaN check: 0


In [ ]:
from sklearn.utils.class_weight import compute_class_weight

class_weights = compute_class_weight(
    'balanced', classes=np.array([0, 1, 2]), y=y_train)
class_weights_dict = {i: float(w) for i, w in enumerate(class_weights)}

print(class_weights_dict)

{0: 0.8042145593869732, 1: 0.698967698967699, 2: 3.068713450292398}


In [ ]:
import joblib, json

np.save(f'{DATA_PROCESSED}/X_train.npy', X_train_p)
np.save(f'{DATA_PROCESSED}/X_val.npy',   X_val_p)
np.save(f'{DATA_PROCESSED}/X_test.npy',  X_test_p)
np.save(f'{DATA_PROCESSED}/y_train.npy', y_train)
np.save(f'{DATA_PROCESSED}/y_val.npy',   y_val)
np.save(f'{DATA_PROCESSED}/y_test.npy',  y_test)

joblib.dump(preprocessor, f'{DATA_PROCESSED}/preprocessor.pkl')

feature_names = preprocessor.get_feature_names_out().tolist()
with open(f'{DATA_PROCESSED}/feature_names.json', 'w') as f:
    json.dump(feature_names, f)

with open(f'{DATA_PROCESSED}/class_weights.json', 'w') as f:
    json.dump(class_weights_dict, f)

print("✓ Semua artifact tersimpan!")

✓ Semua artifact tersimpan!


In [ ]:
import os

files = ['X_train.npy','X_val.npy','X_test.npy',
         'y_train.npy','y_val.npy','y_test.npy',
         'preprocessor.pkl','class_weights.json','feature_names.json']

for f in files:
    path = f'{DATA_PROCESSED}/{f}'
    status = "✓" if os.path.exists(path) else "✗ MISSING"
    print(f"{status}  {f}")

print(f"\nShape train: {X_train_p.shape}")
print(f"Shape val  : {X_val_p.shape}")
print(f"Shape test : {X_test_p.shape}")
print(f"NaN total  : {np.isnan(X_train_p).sum()}")

for name, arr in [('train',y_train),('val',y_val),('test',y_test)]:
    vals, cnts = np.unique(arr, return_counts=True)
    pct = cnts / cnts.sum() * 100
    print(f"{name}: Low={pct[0]:.1f}% Med={pct[1]:.1f}% High={pct[2]:.1f}%")

✓  X_train.npy
✓  X_val.npy
✓  X_test.npy
✓  y_train.npy
✓  y_val.npy
✓  y_test.npy
✓  preprocessor.pkl
✓  class_weights.json
✓  feature_names.json

Shape train: (2099, 23)
Shape val  : (451, 23)
Shape test : (450, 23)
NaN total  : 0
train: Low=41.4% Med=47.7% High=10.9%
val: Low=41.5% Med=47.7% High=10.9%
test: Low=41.3% Med=47.8% High=10.9%
